In [2]:
# https://karel-tavernier.github.io/gerber_writer/html/reference.html
# ! pip install pygerber 
# ! pip install gerber_writer
# curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from gerber_writer import * 

top = DataLayer('Copper,L1,Top')
via_pad = Circle(1, 'ViaPad')
smd_pad = Rectangle(1, 1, 'SMDPad,CuDef')

top.add_pad(via_pad, (0,0))

top.add_trace_arc()
# top.add_traces_path(path, width, function)
# top.add_trace_arc(start: Point, end: Point, center: Point, orientation: str, width: float, function: str, negative: bool = False)
top.add_pad(smd_pad, (0,0))
with open("gerbers\\MyGerber.gbr", 'w') as fo: 
    top.dump_gerber(fo)

In [ ]:
####GERBER_WRITER####

### valid 'functions': See .FileFunction values in the Gerber specification.Comment: None of these have spaces. 
    # Copper,L1,Top,Signal
    # 'Conductor' : 
    # SMDPad,CuDef
    # ThermalReliefPad 
    # Profile,NP
    # Copper,L1,Top
    # Copper,L4,Bot
    # Soldermask,Bot
    # FiducialPad,Local
    # Other,Test
    # TestPad

###DATALAYER(function, negative:bool=False) Define layers ###
top = DataLayer('Copper,L1,Top')

###PATH() Create copper pours or traces ### 
path = Path()

path.moveto( (0,0) ) 
path.lineto( (1,1) ) 
# Path.arcto( (end_x, end_y) , (c_x, c_y) , orientation:'+' | '-' ) 
path.arcto( (1,3) , (1,2) )
#.ADD_TRACES_PATH()
top.add_traces_path(path, width = .254 )
#.ADD_PAD( pad , (x , y) , angle=0 )
top.add_pad(Circle(.254, 'ViaPad') , (4,4) )
top.add_pad(Rectangle(1,3,'SMDPad,CuDef'))
top.add_pad(ChamferedRectangle(1,3, .2 , 'SMDPad,CuDef'))
top.add_pad(RoundedRectangle(2,4, .4, 'SMDPad,CuDef'))



#.ADD_TRACE_LINE()
top.add_trace_line((10,10) , (20,12), 1, 'Conductor') # Q: does the 'start' and 'end' order matter here? A: idts
#.ADD_TRACE_ARC()
top.add_trace_arc()
#.ADD_REGION()
top.add_region()



In [7]:
points_str = "0,1 2,4 5,6"
points = points_str.split() # ["0,1" , "2,4" , '5,6' ]

# points = [ pts.split(',') for pts in points ] # [ ['0', '1'] , ['2' , '4'] ]]
points = [ list(map(float , pts.split(','))) for pts in points_str.split() ]
print(points)

d = {'a':1}
a = d.pop('a')
print(a)
print(d)
print(2**2)

[[0.0, 1.0], [2.0, 4.0], [5.0, 6.0]]
1
{}
4


In [7]:
from gerber_writer import DataLayer 
from gerber_writer import ( Path, set_generation_software, Circle, Rectangle, RoundedRectangle, RoundedThermal)

    
set_generation_software("Robert Driscoll", 'gerber_writer_example.ipynb', '2025.08')
trace_width = .254
via_pad= Circle(.508, 'ViaPad')
top = DataLayer('Copper,L1,Top,Signal', negative=False) # DataLayer 'function' is...

#IC17 footprint
IC17_toe = Rectangle(1.27,2.54, 'SMDPad,CuDef')
top.add_pad(IC17_toe, (65.094, 47.269), 45)
top.add_pad(IC17_toe, (68.047, 50.267), 45)

#Connect one pin to a via 
top.add_trace_line((65.094, 47.269), (65.094+1, 47.269+1), trace_width, 'Conductor')
top.add_pad(via_pad, (65.094+1, 47.269+1))

#Footprint of IC16
IC16_toe = RoundedRectangle(1.257, 2.286, 0.254, 'SMDPad,CuDef')
footprint = ((56.515, 47.879), (60.341, 47.879), (58.428, 43.700))

for toe_location in footprint: 
    top.add_pad(IC16_toe, toe_location)
    
#Connect pin2 to via
top.add_trace_line(footprint[1] , (62.549,47.879), trace_width, 'Conductor') # 'Conductor' is a 'function' (as in purpose, not as in python code function)
top.add_trace_line((62.549, 47.879), (64.350, 49.657), trace_width, 'Conductor')
top.add_pad(via_pad, (64.350, 49.657))

#connect pin3 to IC17 
p1 = (65.000, 43.700)
p2 = (65.000+4.8, 43.700+4.8)
p3 = (68.047, 50.267)
con_3_IC17 = Path()
con_3_IC17.moveto(footprint[2])
con_3_IC17.lineto(p1)
con_3_IC17.lineto(p2)
con_3_IC17.lineto(p3)
top.add_traces_path(con_3_IC17, trace_width, 'Conductor')

#Copper pour, rectangle with one rounded corner 
x_lft = 55
x_rgt = 63
y_bot = 50
y_top = 56
radius = 2.2
pour = Path()
pour.moveto((x_lft, y_bot))
pour.lineto( (x_rgt - radius, y_bot))
pour.arcto( (x_rgt, y_bot +radius) , (x_rgt - radius, y_bot + radius), '+' )
pour.lineto((x_rgt, y_top))
pour.lineto((x_lft, y_top))
pour.lineto((x_lft, y_bot))
top.add_region(pour , 'Conductor')
# Thermal relief pad in copper pour 
top.add_pad(Thermal)
top.add_pad(RoundedThermal(1, 0.8, 0.06, 'ThermalReliefPad', negative=True),
            (x_rgt - radius, y_bot + radius), angle=45)#Embedded via pad in copper pour 
top.add_pad(via_pad, (x_lft + radius, y_top - radius))
#Connect pin1 of IC16 to copper pour 
top.add_trace_line(footprint[0] ,(56.515, 47.879+2.54), trace_width, 'Conductor')

# Connect vias, with arcs, parallel 
# trace_start = (64, 53)
# top.add_pad(via_pad, trace_start)
# connection_a = Path() 
# .moveto(trace_start)
# .lineto( (trace_start[0], trace_start[1]+1) )
# .arcto (trace_start[0]+2 , trace_start[1]+3)
#  What is with the addition of +1, +2 ? Is it because +0 is currently occupied?
import os 
with open('myFile.gbr', 'w') as device:
    top.dump_gerber(device)

In [ ]:
###PCB PROFILE###

from gerber_writer import (DataLayer, Path, set_generation_software)
set_generation_software('Robert Driscoll', 'gerber_writer_example_outline.py', '2025.08')
profile_layer = DataLayer("Profile,NP")
profile = Path()
profile.moveto((0,0))
profile.lineto((150,0))
profile.arcto((160,10), (160,0), '-')
profile.lineto((170, 10))
profile.lineto((170, 90))
profile.lineto((160, 90))
profile.arcto((150, 100), (160, 100), '-')
profile.lineto((0, 100))
profile.lineto((0, 0))
profile_layer.add_traces_path(profile, .5, 'Profile')

with open(os.path.join('gerbers', 'gerber_writer_example_outline.gbr'), 'w') as fo:
    profile_layer.dump_gerber(fo)
    
#User may 

In [ ]:
#User may define a NamedTuple for use with gerber_writer API: 
from typing import NamedTuple 
from gerber_writer import Circle, DataLayer 

class Pt(NamedTuple):
    x: float 
    y: float 
    
via = Circle (.1, 'ViaPad')
copper_btm = DataLayer('Copper,L4,Bot')
origin = Pt(0,0)
copper_btm.add_pad(via, origin)


In [ ]:
top = DataLayer("Copper,L1,Top")

via_pad = Circle(.254 , 'ViaPad')
smd_pad = Rectangle(1,3, 'SMDPad, CuDef')

top.add_pad(via_pad, (1.5, -2.5))
top.add_pad(via_pad, (2.5, -2.5)) 
top.add_pad(smd_pad, (5, -2.5), 45)


In [ ]:
from gerber_writer import DataLayer, Path
top = DataLayer('Copper,L1,Top')        
connection = Path()
connection.moveto((0, 0))
connection.lineto((1, 0))
connection.arcto((1 ,1), (1, 0.5), '+')
top.add_traces_path(connection, 0.1, 'Conductor')
len(top)

top.add_traces_path(connection, .1, 'Conductor') # Is there a function for text? 


In [ ]:
from gerber_writer import DataLayer, Path 
top = DataLayer('Copper,L1,Top')
d_shape = Path() 
d_shape.moveto( (0,0) ) 
d_shape.lineto( (1,0) ) 
d_shape.arcto( (1,1), (1,.5), "+")
d_shape.lineto( ( 0,1) ) 

top.add_trace_arc(start, end, center) # This is a gross way to define a circle, damn you gerber! 

from PySide6.QtGui import QPainterPath
d_shape = QPainterPath()
d_shape.moveTo(0,0) 
d_shape.lineTo(1,0)

# d_shape.arcMoveTo() Q: what is the utility of this function? 
d_shape.arcTo()

import math 
math.tan
